# Graph export core check

합성 입력으로 청크 일치 검증과 CSV 출력을 확인합니다.

In [ ]:
from pathlib import Path
import csv
import json
import tempfile
from payment_card.file_pipeline.graph.export import export_files

work = Path(tempfile.mkdtemp(prefix="graph-export-check-"))
results, chunks, index, out = [work / n for n in ("results", "chunks", "index", "out")]
for folder in (results, chunks, index, out): folder.mkdir()
(index / "documents.csv").write_text("doc_id,sha256\ndoc1," + "a" * 64 + "\n", encoding="utf-8")
(index / "product_docs.csv").write_text("issuer,product_name,product_ids,card_type,doc_id\n", encoding="utf-8")
(chunks / "doc.json").write_text(json.dumps({"doc_id":"doc1", "doc_sha256":"a"*64, "chunks":[{"chunk_id":"c1", "page_no":1}]}), encoding="utf-8")
(results / "c1.result.json").write_text(json.dumps({"chunk_id":"c1", "doc_sha256":"a"*64, "page_no":1, "extraction":{}}), encoding="utf-8")
summary = export_files(results, chunks, index, "synthetic", out)
assert summary["nodes"] == 0 and (out / "fp_nodes.csv").is_file()
print({"nodes": summary["nodes"], "output_files": sorted(p.name for p in out.iterdir())})